# Serializing and Restoring Objects

In this lesson, you will learn to save and restore supported objects and check their types before using their specific operations.

CSC-239 · Module 11 · Lesson 2 of 3

A campus equipment desk keeps a location label in a small card object. You will follow that state into a private file, restore it with a compatible class, and decide whether the restored value is a card the desk can use.

Build on your knowledge of objects, interfaces, primitive records, file paths, and resource cleanup. Each complete example creates and removes its own temporary file. The [Module 11 glossary](terms.md) gathers the terms explained beside the examples.


## Learning Goals

- Serialize and deserialize a supported object using a fixed class agreement.
- Explain marker/version roles and separate the declared Object type, actual type check and reference cast.
- Construct and test a guarded reader that accepts its card type and reports controlled mismatches.


## Why This Matters

The equipment desk wants a later program run to recover the location label `lab` after the current run's variables are gone. Saving the label is only part of the job. The reader also needs a compatible class and a clear rule for accepting the returned value.

This lesson uses one String field, measured as text rather than a numeric quantity. The successful report must name the recovered label. If the file instead contains a different kind of value, the reader must report that mismatch and skip the card's getter. It closes each stream and removes only its own temporary file.

Programs that preserve settings or small records face the same separation between storing state and safely using recovered values. Here we read only controlled files created by the examples. These practice classes do not replace the project's supplied input/output interface or shared classes.


## Check Your Starting Point

Retrieve the class and file ideas that the new object-stream operations will build on.


Explain how an object differs from a reference to it, and how a class can implement an interface. Recall what final prevents and what long represents. Contrast the previous primitive field schema with saving an object representation. Identify the separate jobs of stream closing and file deletion.


In [ ]:
Your response:

Objects and references:
Interface agreement:
final and long:
Field schema and object representation:
Closing and deletion:


<details>
<summary>Show answer</summary>

An object holds state and exposes operations through its class. A reference identifies an object; another reference can identify the same object without copying it. An interface describes a type agreement that a class can implement.

The previous lesson wrote each primitive field in an agreed type, order, and meaning. The new lesson writes a supported object's representation, but the writer and reader still need an agreement about its class.

A final variable cannot be reassigned after initialization. A long holds whole numbers over a wider range than int. Neither fact alone tells us what a particular field means. Try-with-resources closes each stream, and the outer finally removes the example's owned file after those resource blocks.

</details>


## Video Demonstration

Watch a supported card move through writing, reading, a type check, and access to its label. The connected reading below teaches the same operations in more detail.

<video controls preload="metadata" width="960" style="max-width:100%;height:auto;">
<source src="media/02_serializing_and_restoring_objects/demo.mp4" type="video/mp4">
<track kind="captions" src="media/02_serializing_and_restoring_objects/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the video transcript](media/02_serializing_and_restoring_objects/transcript.md).


## Concept

### Save the state of a supported object

A campus equipment desk uses a `LabelCard` object to hold the label `lab`. The desk wants to save that card's state and restore a card whose label has the same meaning later. This extends the previous lesson's primitive records: the saved value now belongs to an object with a class and a getter.

**Object serialization** writes a representation of supported object state. An **ObjectOutputStream** provides **`writeObject`** for that operation, wrapping the file's byte output stream. After the class has been declared and the writer opened, our example performs:

```java
        writer.writeObject(new LabelCard("lab"));
```

The constructor creates a LabelCard with its label set to `lab`. `writeObject` then writes the supported representation through the stream. This does not save the Java source file, the notebook kernel, or a currently running method. The complete example closes the writer before opening its reader.

Not every object can be written this way. The class must support serialization, and relevant referenced state must also meet the serialization rules. We will make that support explicit in the class declaration.


### Restore state using a compatible class

**Object deserialization** reads a saved representation and reconstructs the corresponding object state. An **ObjectInputStream** supplies the operation:

```java
        Object value = reader.readObject();
```

This fragment assumes the input stream has been opened on the file written by our example. The method **`readObject`** restores the saved value. Its declared return type is `Object`, so that is the type of the variable we use initially. We will explain how to check and use its more specific type below.

The reader needs compatible class definitions. The saved file is not a substitute for the program's classes. **ClassNotFoundException** can report that a required class could not be found, and I/O exceptions can report other reading or format problems.

Our fixture writes and reads a file it just created using the same class definition. That controlled agreement lets us study state restoration. We should not describe the result as the same in-memory object as the one originally constructed; the lesson verifies the restored label's value.


### Declare support with a marker interface

A **marker interface** identifies a capability without requiring the implementing class to supply interface methods. Java's **Serializable** interface marks a class as supporting this serialization mechanism. Our declaration uses the familiar `implements` keyword:

```java
class LabelCard implements Serializable {
```

This is the opening line of the complete class, whose closing brace appears later. LabelCard still has its ordinary state and behavior: a private String field, a constructor that sets it, and a getter that reads it. Serializable does not require us to write a method named `serialize`.

The marker does not mean that every possible field or connected object is automatically suitable for storage. In this example, the label is a String, which supports serialization. Keeping the example's state small makes the support agreement explicit rather than hiding it inside a large object structure.


### Give the class a serialization version identifier

A **serialization version identifier** participates in checking whether a saved representation and a local class definition are compatible. Our class declares it with the conventional field name **`serialVersionUID`**:

```java
    private static final long serialVersionUID = 1L;
```

The field belongs to the class because it is `static`. The keyword **`final`** prevents reassignment after initialization. **`long`** is the primitive whole-number type used for this identifier, and the `L` suffix makes `1L` a long literal. `private` keeps ordinary direct field access inside the class.

This number is a compatibility identifier, not the number of cards saved or the size of the file. An explicit value avoids relying on a generated identifier, but matching it does not guarantee compatibility with every arbitrary change to the class.

Our write and read use the same class agreement. The lesson is not an exercise in changing old saved formats; it establishes why the identifier appears and why it should not be treated as a universal repair for incompatible data.


Follow supported LabelCard state into an object-stream representation. The labeled boxes show a conceptual view, not a literal byte dump. These are the already-explained `lab` examples.

<details class="animation-panel" open>
<summary>Show or hide animation: Save supported card state</summary>

<p><img src="media/02_serializing_and_restoring_objects/serialize_supported_state.gif" alt="The lab label is saved in an object representation while its class supplies methods and compatibility information." width="960" style="max-width:100%;height:auto;"></p>

</details>

[View still: Save supported card state](media/02_serializing_and_restoring_objects/serialize_supported_state_still.png). The sequence lasts 10 seconds. Hide it to stop visible motion; the still and prose retain the explanation.


### Understand what an Object reference exposes

**Object** is Java's common superclass type for class instances. A reference declared as Object can hold the LabelCard restored by this example, but its declared type does not expose LabelCard's particular getter.

The variable `value` therefore has two relevant descriptions: its declared type is Object, while the restored non-null object's runtime class in the successful case is LabelCard. Declaring the variable as Object did not remove the label or turn the card into a different kind of object.

The compiler uses the declared type to decide which method calls are permitted through a reference. Before using `getLabel`, the program must establish that the restored value can be used as a LabelCard and obtain a reference of that type. Separating these steps makes a wrong-object case explicit.


### Check the restored value before using a specific type

A **runtime type check** asks whether an actual value is compatible with a requested reference type. The keyword **`instanceof`** performs that test:

```java
        if (value instanceof LabelCard) {
```

This fragment opens the success branch of our conditional. For a non-null LabelCard value, the condition is true and the program enters the branch. A String value instead fails this LabelCard check. A null reference also produces false.

The check does not change the object or repair the wrong type. It selects which path the program can safely take. In the complete example, the else branch reports `Unexpected object type.` rather than attempting the LabelCard getter on an incompatible object.

Check the type you intend to use. A condition that proves a value is a String would not justify treating that same value as a LabelCard. The later debugging tasks use this distinction to connect the guard to the operation it is supposed to protect.


### Use a checked cast to access the card's operation

A **reference cast** tells Java to use a reference through a specified compatible type. Inside the successful LabelCard branch, the program uses:

```java
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
```

The type in parentheses requests the cast. The assignment stores the resulting LabelCard reference in `restored`. Both references identify the same restored object; the cast does not construct another card, rewrite its fields, or convert a String into a card.

The getter call is now available through the declared LabelCard type. It returns the stored `lab` string, so the program prints `Label: lab`. The preceding type check supports the cast, while the output verifies the restored state.

A cast applied to an incompatible non-null object can throw **ClassCastException**. The complete reader uses a matching guard so that its wrong-type path remains a deliberate report. A null reference can be cast, but calling a method through it would still fail; our `instanceof` guard excludes that case as well.

The full example now has a clear sequence: declare supported state, write it, read an Object value, check its type, use the checked cast, and report the field. Practice will keep those roles separate while changing the card data and testing a mismatched object.


Separate reconstruction from the Object result, type check and reference cast. The labeled boxes show a conceptual view, not a literal byte dump. These are the already-explained `lab` examples.

<details class="animation-panel" open>
<summary>Show or hide animation: Restore, check, and access the card</summary>

<p><img src="media/02_serializing_and_restoring_objects/restore_and_access.gif" alt="Deserialization reconstructs a LabelCard; an Object reference passes the type check and the cast accesses that same restored object." width="960" style="max-width:100%;height:auto;"></p>

</details>

[View still: Restore, check, and access the card](media/02_serializing_and_restoring_objects/restore_and_access_still.png). The sequence lasts 12.5 seconds. Hide it to stop visible motion; the still and prose retain the explanation.


### Follow a complete unexpected-type comparison

This complete example declares TypeCard but writes the String `lab` directly. The actual argument to writeObject determines the saved value. Merely declaring a card class does not put a card into the file.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class TypeCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public TypeCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("type-check-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("lab");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof TypeCard) {
            TypeCard restored = (TypeCard) value;
            System.out.println(restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

It prints `Unexpected object type.` The returned String fails the TypeCard check, so the cast and getter do not execute. This is a deliberate application report about an unexpected value, not a failure to read a valid String representation.

Follow the actual stored String through the expected-card check and rejected branch. The labeled boxes show a conceptual view, not a literal byte dump. These are the already-explained `lab` examples.

<details class="animation-panel" open>
<summary>Show or hide animation: Reject an unexpected stored type</summary>

<p><img src="media/02_serializing_and_restoring_objects/reject_unexpected_type.gif" alt="A serialized String fails the TypeCard check and prints Unexpected object type without attempting a card cast." width="960" style="max-width:100%;height:auto;"></p>

</details>

[View still: Reject an unexpected stored type](media/02_serializing_and_restoring_objects/reject_unexpected_type_still.png). The sequence lasts 12.5 seconds. Hide it to stop visible motion; the still and prose retain the explanation.


## Worked Example

### Connect class support, file work, and the report

The complete LabelCard program below gathers the statements from the reading into one runnable example. Its first three imports supply the object input stream, object output stream, and serialization marker. Files and Path supply the familiar file operations and location type.

The class declares its fixed version identifier and one private String field. Its constructor assigns the parameter to that field with `this.label = label`; its getter returns the field. The class definition stays available while this example writes and reads.

```java
Path file = Files.createTempFile("label-card-", ".bin");
```

This call creates a new empty temporary file and returns its Path. The requested prefix and suffix help name it; generated characters distinguish the file. Keep that returned path for writing, reading, and deletion. The `.bin` suffix does not choose a format by itself.

### Write, close, and then read

```java
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("lab"));
    }
```

Files opens the byte output, and ObjectOutputStream wraps it with object-writing support. The write receives a newly constructed LabelCard. The resource block closes the writer before the next block opens byte input and wraps it in ObjectInputStream.

The reader assigns the returned value to Object. The LabelCard check selects either the cast and getter or the mismatch report. The cast uses the same restored object through a LabelCard reference; the report adds its console label to the String returned by the getter.

### Finish the owned fixture

```java
} finally {
    Files.deleteIfExists(file);
}
```

The reader's resource block closes before this outer finally removes the temporary file. Closing and deleting are different operations. Deletion can itself fail, so the example does not promise cleanup under every possible file-system failure.

The Java notebook permits these checked calls at the top level. In a conventional method, handle or declare their checked exceptions as in Module 7. The video uses a main method that declares the I/O and class-lookup exceptions. Running the complete notebook cell again creates a new file instead of relying on the deleted one.


In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("label-card-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("lab"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}

Expected output:

```text
Label: lab
```

The writer received a LabelCard holding `lab`. The reader restored that state with the available class definition. Its value passed the LabelCard check, so the cast made the getter available and the getter supplied the saved text.

This report verifies a restored field value. It does not claim that the restored card is the original in-memory instance, that source code was saved, or that any future class change will be compatible.


## Guided Practice

First predict and observe a changed value. Then complete missing operations, modify a boundary value, and repair a guard. Keep each answer closed until after your attempt. Writing cells are for predictions and explanations; blank Java cells are for complete programs.


### Predict a changed location

Read the complete program below without running it. Predict every printed line for the changed label `studio`. Explain which argument determines the stored value, why readObject initially gives an Object result, and why the guard comes before the cast.


In [ ]:
Your response:

Prediction before run:
Stored argument and declared result:
Guard before cast:


In [ ]:
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("11-02-label-studio-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("studio"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}

Run the complete changed-label program. Retain your prediction, record the actual output, and explain any difference. Explain why value and restored refer to the same reconstructed object after the cast.


In [ ]:
Your response:

Actual output:
Comparison and same-object explanation:


<details>
<summary>Show answer</summary>

### A changed location label

The changed String is ordinary saved instance state. The class/version agreement stays fixed. The read value is a LabelCard, so its guard permits the same-object reference cast and getLabel returns studio.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("11-02-label-studio-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("studio"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Label: studio
```

The stored constructor argument supplies the changed text. The cast does not create the text or a second restored object.

</details>


### Complete the support and reader agreement

Copy this intentionally incomplete template into the blank Java cell, then replace MARKER, VERSION, READ_VALUE, and both EXPECTED_TYPE positions. Choose from Serializable, 1L, readObject, and LabelCard. Before running, explain each choice and reconstruct the already-taught `lab` result.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements MARKER {
    private static final long serialVersionUID = VERSION;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("label-card-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("lab"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.READ_VALUE();
        if (value instanceof EXPECTED_TYPE) {
            LabelCard restored = (EXPECTED_TYPE) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```


In [ ]:
Your response:

Marker:
Version identifier:
Read operation:
Guard and cast type:
Reconstructed output:


Run your completed template. Record the actual output and explain why the guard and cast use matching type names. State one limit of what the version identifier guarantees.


In [ ]:
Your response:

Actual output:
Matching type names:
Version limit:


<details>
<summary>Show answer</summary>

Use Serializable for MARKER, 1L for VERSION, readObject for READ_VALUE, and LabelCard in both EXPECTED_TYPE positions. The marker declares support; it does not require a new method. The identifier participates in class compatibility checking but cannot guarantee every possible class change.

The guard must check the type required by the cast. After that successful cast, the LabelCard reference exposes getLabel.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("label-card-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("lab"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Label: lab
```

</details>


### Save an empty label

Make a complete copy of the studio program in the Java cell. Change only the saved label argument to the empty String `""`; keep the class, guard, cast, and cleanup. Before running, predict the exact output, including what remains after the colon. Explain whether an empty field changes the card type.


In [ ]:
Your response:

Prediction before run:
Empty field and card type:


Run the complete empty-label variant. Record its actual output and compare it with your prediction. Explain the difference between changing saved field text and changing the class agreement.


In [ ]:
Your response:

Actual output:
Field value versus class agreement:


<details>
<summary>Show answer</summary>

### An empty saved label

An empty String is still a String value saved in the card. The guard checks the card type rather than label length. The output has the Label: prefix followed by its space and newline; emptiness does not imply a wrong type.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("11-02-label-empty-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard(""));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Label: 
```

The output retains the space after the colon and then ends the line. A card whose field is empty remains a LabelCard; its field contents do not change its runtime type.

</details>


### Repair a guard that protects the wrong type

The first complete diagnostic below deliberately checks String before a LabelCard cast. The second changes its stored value to a LabelCard but retains that same wrong guard. Before running either, predict both behaviors and identify the first operation that fails or takes the wrong branch. Then repair the guard in complete copies in the Java work cell, testing both inputs.

Intentionally faulty String case:

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("11-02-wrong-guard-string-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof String) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Intentionally faulty card case:

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("11-02-wrong-guard-card-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("lab"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof String) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```


In [ ]:
Your response:

Faulty String prediction and first failure:
Faulty card prediction and wrong branch:
Proposed guard repair:


Record the actual results of both repaired cases: stored String `status` and stored LabelCard `lab`. Explain how the repair protects the cast and why a reader that always rejects would still be incorrect.


In [ ]:
Your response:

Repaired String actual output:
Repaired card actual output:
Why both tests matter:


<details>
<summary>Show answer</summary>

With the faulty String guard, the stored String passes the check but the following LabelCard cast throws ClassCastException before any print. With a stored LabelCard, the same faulty guard is false and wrongly selects the mismatch report. These are different failures of the same incorrect condition.

Repair the guard to check LabelCard. Test both accepted and rejected values: a reader that always rejects would pass only the rejection test.

### Repaired reader: stored String

The String is valid serialized data but is not a LabelCard. The repaired instanceof LabelCard condition is false, so the cast and getLabel call are skipped and the mismatch branch prints once.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("11-02-repaired-string-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

### Repaired reader: stored LabelCard

The stored value is a LabelCard, so the same repaired condition succeeds. Casting accesses that restored card and its label. An always-false guard would fail this accepted case even if it passed the String rejection case.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class LabelCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String label;
    public LabelCard(String label) { this.label = label; }
    public String getLabel() { return label; }
}
Path file = Files.createTempFile("11-02-repaired-card-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new LabelCard("lab"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof LabelCard) {
            LabelCard restored = (LabelCard) value;
            System.out.println("Label: " + restored.getLabel());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Label: lab
```

Both complete fixtures close their streams and attempt deletion in finally. A successful type check is a condition for the cast; it does not rewrite an incompatible value.

</details>


## Independent Practice

### Build a guarded equipment tag reader

The desk now needs a TagCard with one saved String tag. Construct a complete round trip instead of filling a supplied program. Keep the hidden answer closed while designing the class and checking both useful and unexpected input.


Define TagCard with one private String tag, a constructor, and getTag. Mark it Serializable and use the fixed version identifier 1L. Write a card with tag `loaner` to a fresh private temporary file. Close the writer, read Object, check TagCard before its matching cast, and print the recovered field with the label `Tag: `. Otherwise print `Unexpected object type.` Close the reader and delete your file in finally. Include all imports and setup. Before coding or running, outline the sequence and predict the baseline output.


In [ ]:
Your response:

Class and file-work design:
Baseline prediction before run:


Run your complete baseline. Record its output and explain the check-before-cast order. Identify where the writer and reader close and where the file is deleted. Explain why the whole program recreates its own setup.


In [ ]:
Your response:

Baseline actual output:
Check and cast:
Close and delete:
Fresh setup:


### Test four separate cases

Preserve your baseline, then prepare four complete variants: a TagCard with tag `spare`, a TagCard with an empty tag, the String `"status"` written directly, and null written directly. Keep the TagCard guard and matching cast unchanged.

Write all four named predictions in the response cell before the first case run. Next run all four complete versions with their own fresh file setup and cleanup. After the last run, record each named actual output and explanation in the same response cell. Explain which cases are accepted, which are rejected, and why an empty tag differs from null. Keep the answer closed until these observations are recorded. Use your Java work cell above for each complete variant.


In [ ]:
Your response:

ALL PREDICTIONS — complete before the first case run
Spare tag prediction:
Empty tag prediction:
String status prediction:
Null prediction:

ALL OBSERVATIONS — complete after the last case run
Spare tag actual output and explanation:
Empty tag actual output and explanation:
String status actual output and explanation:
Null actual output and explanation:
Empty tag versus null:


<details>
<summary>Show answer</summary>

### Baseline: loaner tag

The TagCard class marks serialization support and uses the fixed version identifier. Its ordinary String tag is saved and restored. The actual TagCard passes the TagCard check before the cast and getTag call. Changing or emptying this String does not change the supported object type.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class TagCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String tag;
    public TagCard(String tag) { this.tag = tag; }
    public String getTag() { return tag; }
}
Path file = Files.createTempFile("11-02-tag-loaner-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new TagCard("loaner"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof TagCard) {
            TagCard restored = (TagCard) value;
            System.out.println("Tag: " + restored.getTag());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Tag: loaner
```

### Changed field: spare tag

The TagCard class marks serialization support and uses the fixed version identifier. Its ordinary String tag is saved and restored. The actual TagCard passes the TagCard check before the cast and getTag call. Changing or emptying this String does not change the supported object type.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class TagCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String tag;
    public TagCard(String tag) { this.tag = tag; }
    public String getTag() { return tag; }
}
Path file = Files.createTempFile("11-02-tag-spare-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new TagCard("spare"));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof TagCard) {
            TagCard restored = (TagCard) value;
            System.out.println("Tag: " + restored.getTag());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Tag: spare
```

### Boundary: empty tag

The TagCard class marks serialization support and uses the fixed version identifier. Its ordinary String tag is saved and restored. The actual TagCard passes the TagCard check before the cast and getTag call. Changing or emptying this String does not change the supported object type.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class TagCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String tag;
    public TagCard(String tag) { this.tag = tag; }
    public String getTag() { return tag; }
}
Path file = Files.createTempFile("11-02-tag-empty-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(new TagCard(""));
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof TagCard) {
            TagCard restored = (TagCard) value;
            System.out.println("Tag: " + restored.getTag());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Tag: 
```

### Unexpected type: String status

readObject returns a String rather than a TagCard; instanceof TagCard is false. The mismatch branch prints once and the cast/getter are skipped. A valid serialized value does not necessarily satisfy this reader’s application type requirement.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class TagCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String tag;
    public TagCard(String tag) { this.tag = tag; }
    public String getTag() { return tag; }
}
Path file = Files.createTempFile("11-02-tag-string-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject("status");
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof TagCard) {
            TagCard restored = (TagCard) value;
            System.out.println("Tag: " + restored.getTag());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

### Boundary: null value

The stored value is null; instanceof TagCard is false for null, so no card reference is dereferenced. The mismatch branch prints once and the cast/getter are skipped. A valid serialized value does not necessarily satisfy this reader’s application type requirement.

```java
import java.io.ObjectInputStream;
import java.io.ObjectOutputStream;
import java.io.Serializable;
import java.nio.file.Files;
import java.nio.file.Path;
class TagCard implements Serializable {
    private static final long serialVersionUID = 1L;
    private String tag;
    public TagCard(String tag) { this.tag = tag; }
    public String getTag() { return tag; }
}
Path file = Files.createTempFile("11-02-tag-null-", ".bin");
try {
    try (ObjectOutputStream writer = new ObjectOutputStream(Files.newOutputStream(file))) {
        writer.writeObject(null);
    }
    try (ObjectInputStream reader = new ObjectInputStream(Files.newInputStream(file))) {
        Object value = reader.readObject();
        if (value instanceof TagCard) {
            TagCard restored = (TagCard) value;
            System.out.println("Tag: " + restored.getTag());
        } else {
            System.out.println("Unexpected object type.");
        }
    }
} finally {
    Files.deleteIfExists(file);
}
```

Expected output:

```text
Unexpected object type.
```

The first three cases pass the TagCard check because the containing object has the expected type. The last two skip the cast and getter. An empty String inside a card is different from a null value returned by readObject. Each complete test declares its class, creates its own file, closes the writer before reading, and cleans up afterward.

</details>


## Summary

Serialization writes supported object state; deserialization reconstructs saved state using compatible class definitions. The marker interface and version identifier support that agreement, but they do not make arbitrary changes compatible.

Keep three reader steps separate: receive the declared Object result, check the actual value's type, and cast only in the matching branch. The cast changes the access type of a reference, while the getter retrieves the restored field. Test accepted values as well as mismatches, then close resources and remove the owned file.


Close the answers. Explain object serialization, object deserialization, marker interface, serialization version identifier, declared Object type, runtime type check, and reference cast in your own words. Explain how the last three cooperate in a guarded reader.


In [ ]:
Your response:

Serialization:
Deserialization:
Marker:
Version identifier:
Object:
Type check:
Reference cast:
Reader sequence:


<details>
<summary>Show answer</summary>

Object serialization writes a supported representation, and object deserialization reconstructs its saved state with compatible classes. Serializable is a marker interface: it declares support without requiring a serialize method. serialVersionUID supplies a compatibility identifier, not a universal guarantee.

Object is the declared type returned by readObject. The actual returned value may have a more specific class or may be null. instanceof tests compatibility with the type the reader needs, returning false for null. A reference cast inside the matching branch allows that type's operations on the same restored object. A wrong cast can throw ClassCastException; it cannot convert one kind of object into another.

</details>


## Reflection

Apply the same agreement to a small object from another subject or activity. In the next lesson, you will decide which fields should be saved and which represent temporary state. That choice builds on today's supported-class and checked-reader sequence.


Describe a small domain object and one field worth saving. State the class-specific method a reader would use, the type check that should precede it, and an unexpected value to test. Explain why checking the restored type and keeping compatible class definitions are separate responsibilities.


In [ ]:
Your response:

Object and saved field:
Specific method and guard:
Unexpected-value test:
Type check versus compatible definition:


## Supplemental Reading

- [ObjectOutputStream and writing objects](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/ObjectOutputStream.html) documents supported state and writeObject.
- [ObjectInputStream and reading objects](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/ObjectInputStream.html) documents readObject, class resolution, and its declared return type.
- [Serializable and version identifiers](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/io/Serializable.html) explains the marker and serialVersionUID agreement.
- [Java Object](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Object.html) describes the common superclass and its operations.
- [Java cast and instanceof expressions](https://docs.oracle.com/javase/specs/jls/se21/html/jls-15.html) specifies reference casts and type checks in sections 15.16 and 15.20.2.
- [Files temporary files and byte streams](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/nio/file/Files.html) documents createTempFile, the stream-opening methods, and deleteIfExists.
